In [6]:
import meep as mp
import argparse

def main(N, sy, fcen, df):
    resolution = 20   # pixels/um

    eps = 13          # dielectric constant of waveguide
    w = 1.2           # width of waveguide
    r = 0.36          # radius of holes
    d = 1.4           # defect spacing (ordinary spacing = 1)


    pad = 2           # padding between last hole and PML edge
    dpml = 1          # PML thickness
    
    sx = 2*(pad+dpml+N)+d-1  # size of cell in x direction
    cell = mp.Vector3(sx,sy,0)
    
    blk = mp.Block(size=mp.Vector3(mp.inf,w,mp.inf), material=mp.Medium(epsilon=eps))
    geometry = [blk]

    for i in range(N):
            geometry.append(mp.Cylinder(r, center=mp.Vector3(d/2+i)))
            geometry.append(mp.Cylinder(r, center=mp.Vector3(-(d/2+i))))
    
    pml_layers = [mp.PML(1.0)]
    
    src = [mp.Source(mp.GaussianSource(fcen, fwidth=df),
                     component=mp.Ey,
                     center=mp.Vector3(-0.5*sx+dpml),
                     size=mp.Vector3(0,w))]
    
    sym = [mp.Mirror(mp.Y, phase=-1)]
    
    sim = mp.Simulation(cell_size=cell,
                        geometry=geometry,
                        boundary_layers=pml_layers,
                        sources=src,
                        symmetries=sym,
                        resolution=resolution)
    
    freg = mp.FluxRegion(center=mp.Vector3(0.5*sx-dpml-0.5),
                         size=mp.Vector3(0,2*w))

    nfreq = 500 # number of frequencies at which to compute flux

    # transmitted flux
    trans = sim.add_flux(fcen, df, nfreq, freg)
    
    vol = mp.Volume(mp.Vector3(0), size=mp.Vector3(sx))

    sim.run(mp.at_beginning(mp.output_epsilon),
                mp.during_sources(mp.in_volume(vol, mp.to_appended("hz-slice", mp.at_every(0.4, mp.output_hfield_z)))),
                until_after_sources=mp.stop_when_fields_decayed(50, mp.Ey, mp.Vector3(0.5*sx-dpml-0.5), 1e-3))

    sim.display_fluxes(trans)  # print out the flux spectrum

In [7]:
N = 3                    # number of holes on either side of defect
sy = 6                   # size of cell in y direction (perpendicular to wvg.)
fcen = 0.25              # pulse center frequency
df = 0.2                 # pulse frequency width
main(N, sy, fcen, df)

-----------
Initializing structure...
field decay(t = 50.025000000000006): 3.9579082047198376e-05 / 3.9579082047198376e-05 = 1.0
field decay(t = 100.05000000000001): 6.06240189482104e-05 / 6.06240189482104e-05 = 1.0
field decay(t = 150.07500000000002): 1.7734570553698603e-05 / 6.06240189482104e-05 = 0.2925337326918034
field decay(t = 200.10000000000002): 7.46802529725916e-06 / 6.06240189482104e-05 = 0.12318591586016277
field decay(t = 250.125): 5.4396330913507075e-06 / 6.06240189482104e-05 = 0.08972735865627206
field decay(t = 300.15000000000003): 4.232911651741274e-06 / 6.06240189482104e-05 = 0.06982235300759829
field decay(t = 350.175): 3.47662884582435e-06 / 6.06240189482104e-05 = 0.057347383201274534
field decay(t = 400.20000000000005): 2.847134687922054e-06 / 6.06240189482104e-05 = 0.046963806381003725
field decay(t = 450.225): 2.3289750979705433e-06 / 6.06240189482104e-05 = 0.038416705760799676
field decay(t = 500.25): 1.9177580551626054e-06 / 6.06240189482104e-05 = 0.03163363446